# NB27: Advanced Census — MOE Propagation & NAICS/SOC Crosswalks

This notebook covers two advanced data tools:

1. **Margin of Error (MOE) Propagation** — correctly propagate ACS margins of error
   through sums, differences, proportions, and ratios
2. **NAICS/SOC Crosswalks** — classify industries and occupations using
   standard federal classification codes

## Part 1: Margin of Error Propagation

ACS estimates come with margins of error (MOE) at 90% confidence.
When you combine estimates (sum, subtract, divide), the MOE must be
propagated correctly — you can't just add them.

In [ ]:
from siege_utilities.data.moe_propagation import (
    Estimate,
    moe_sum,
    moe_difference,
    moe_proportion,
    moe_ratio,
    moe_product,
    flag_unreliable,
)

### 1.1 The Estimate Object

In [ ]:
# An ACS estimate with its margin of error
pop_tract_a = Estimate(value=5200, moe=450)
pop_tract_b = Estimate(value=3800, moe=380)
pop_tract_c = Estimate(value=150, moe=120)  # Small population — high CV

for name, est in [("Tract A", pop_tract_a), ("Tract B", pop_tract_b), ("Tract C", pop_tract_c)]:
    lo, hi = est.confidence_interval()
    print(f"{name}: {est.value:,} +/- {est.moe} (CV={est.cv_pct:.1f}%, "
          f"90% CI: [{lo:,.0f}, {hi:,.0f}], reliable={est.is_reliable()})")

### 1.2 Summing Estimates

In [ ]:
# Combine tracts into a larger geography
combined = moe_sum([pop_tract_a, pop_tract_b, pop_tract_c])
print(f"Combined: {combined.value:,} +/- {combined.moe:.0f}")
print(f"CV: {combined.cv_pct:.1f}% (improved by aggregation)")
print(f"\nNote: MOE({combined.moe:.0f}) < simple sum({pop_tract_a.moe + pop_tract_b.moe + pop_tract_c.moe})")
print("MOE propagation uses sqrt(sum of squares), not simple addition.")

### 1.3 Differences and Proportions

In [ ]:
# Difference: how many more people in Tract A than B?
diff = moe_difference(pop_tract_a, pop_tract_b)
print(f"Difference (A - B): {diff.value:,} +/- {diff.moe:.0f}")
lo, hi = diff.confidence_interval()
print(f"90% CI: [{lo:,.0f}, {hi:,.0f}]")
print(f"Statistically significant? {'Yes' if lo > 0 or hi < 0 else 'No (CI includes 0)'}")

print()

# Proportion: what fraction of the combined area is in Tract A?
prop = moe_proportion(pop_tract_a, combined)
print(f"Proportion (A / total): {prop.value:.3f} +/- {prop.moe:.3f}")
print(f"  = {prop.value * 100:.1f}% +/- {prop.moe * 100:.1f} percentage points")

### 1.4 Flagging Unreliable Estimates

In [ ]:
# Census Bureau considers CV > 40% unreliable
estimates = [pop_tract_a, pop_tract_b, pop_tract_c]
unreliable = flag_unreliable(estimates, cv_threshold=0.40)

print(f"Unreliable estimates (CV > 40%): {len(unreliable)} of {len(estimates)}")
for idx, est in unreliable:
    print(f"  Index {idx}: {est.value:,} +/- {est.moe} (CV={est.cv_pct:.1f}%)")

## Part 2: NAICS/SOC Classification Crosswalks

Federal data uses two classification systems:
- **NAICS** (North American Industry Classification System) — classifies businesses
- **SOC** (Standard Occupational Classification) — classifies workers

siege_utilities provides parsing, hierarchy traversal, and fuzzy matching.

In [ ]:
from siege_utilities.data.naics_soc_crosswalk import (
    parse_naics,
    naics_ancestors,
    naics_to_sector,
    parse_soc,
    soc_to_major_group,
    fuzzy_match_naics,
    NAICS_SECTORS,
    SOC_MAJOR_GROUPS,
)

### 2.1 NAICS Code Parsing

In [ ]:
# Parse a NAICS code
code = parse_naics("541511")
print(f"Code: {code.code}")
print(f"Title: {code.title}")
print(f"Level: {code.level} (6-digit = national industry)")
print(f"Sector: {code.sector}")

# Walk up the hierarchy
print(f"\nAncestors of {code.code}:")
for ancestor in naics_ancestors(code.code):
    a = parse_naics(ancestor)
    print(f"  {a.code}: {a.title} (level {a.level})")

### 2.2 Sector Overview

In [ ]:
print(f"NAICS has {len(NAICS_SECTORS)} sectors:")
for code, title in sorted(NAICS_SECTORS.items()):
    print(f"  {code}: {title}")

### 2.3 SOC Codes

In [ ]:
# Parse an SOC code
soc = parse_soc("15-1252")
print(f"SOC: {soc.code} — {soc.title}")
print(f"Level: {soc.level}")
print(f"Major group: {soc.major_group}")

major_code, major_title = soc_to_major_group("15-1252")
print(f"Major group: {major_code} — {major_title}")

print(f"\nSOC has {len(SOC_MAJOR_GROUPS)} major groups:")
for code, title in sorted(SOC_MAJOR_GROUPS.items())[:5]:
    print(f"  {code}: {title}")
print(f"  ... and {len(SOC_MAJOR_GROUPS) - 5} more")

### 2.4 Fuzzy Matching

In [ ]:
# Match free-text to NAICS sectors
for text in ["software consulting", "restaurant food", "construction building", "health care"]:
    matches = fuzzy_match_naics(text, threshold=0.3)
    if matches:
        best = matches[0]
        print(f"  '{text}' → {best[0]}: {best[1]} (score={best[2]:.2f})")
    else:
        print(f"  '{text}' → no match")

## Summary

### MOE Propagation
| Function | Use Case |
|----------|----------|
| `moe_sum()` | Aggregate tracts → county, combine racial groups |
| `moe_difference()` | Year-over-year change, geographic comparison |
| `moe_proportion()` | Percentage of total (nested estimates) |
| `moe_ratio()` | Non-nested ratio (e.g., renters per owner) |
| `flag_unreliable()` | QA — flag high-CV estimates before publishing |

### NAICS/SOC
| Function | Use Case |
|----------|----------|
| `parse_naics()` / `parse_soc()` | Parse code into structured object |
| `naics_ancestors()` | Walk up the hierarchy |
| `naics_to_sector()` | Roll up to 2-digit sector |
| `fuzzy_match_naics()` | Match free text to industry codes |